In [ ]:
import sys; sys.path.append('..')
import parametrization, sparse_matrices, mesh, numpy as np, importlib, pickle, wall_generation, matplotlib, inflation
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf, mesh_utilities, fd_validation
from matplotlib import pyplot as plt

In [ ]:
# Choose reasonable stretching bounds
alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(2, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
print(alphaMin, alphaMax)

In [ ]:
lilium = mesh.Mesh("../../examples/lilium.msh")
uv = np.loadtxt('../MetricFittingExperiments/data/lilium_param.txt')

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(lilium, uv)
rparam.alphaMin = alphaMin
rparam.alphaMax = alphaMax
rparam.bendRegW = 1.0

Verify that the IDT-based dual Laplacian alpha regularization term gives the same result as evaluating the Laplacian's quadratic form on the vector of alphas.

In [ ]:
rparam.dualLaplacianStencil.type = rparam.dualLaplacianStencil.Type.DualMeshIDT
alphaReg = rparam.energy(rparam.EnergyType.AlphaRegularization)
positiveSemidefL = -mesh_utilities.barycentricDualIDTLaplacian(lilium)
alphas = rparam.getAlphas()
np.abs(0.5 * alphas.dot(positiveSemidefL @ alphas) - alphaReg) / alphaReg

Run finite difference tests for all settings of the dual Laplacian regularization

In [ ]:
def dualLapRegFDValidation(name, offset):
    perturb = np.random.uniform(low=-1,high=1, size=rparam.numVars())
    plt.subplot(3, 4, offset + 1)
    fd_validation.gradConvergencePlot(rparam, perturb=perturb, energyType=rparam.EnergyType.AlphaRegularization)
    plt.title(f'{name}, alpha gradient')
    plt.subplot(3, 4, offset + 2)
    fd_validation.gradConvergencePlot(rparam, perturb=perturb, energyType=rparam.EnergyType.PhiRegularization)
    plt.title(f'{name}, phi gradient')
    
    plt.subplot(3, 4, offset + 3)
    fd_validation.hessConvergencePlot(rparam, perturb=perturb, energyType=rparam.EnergyType.AlphaRegularization)
    plt.title(f'{name}, alpha Hessian')
    plt.subplot(3, 4, offset + 4)
    fd_validation.hessConvergencePlot(rparam, perturb=perturb, energyType=rparam.EnergyType.PhiRegularization)
    plt.title(f'{name}, phi Hessian')
    
fig = plt.figure(figsize=(16, 10))
rparam.dualLaplacianStencil.type = rparam.dualLaplacianStencil.Type.DualGraph
dualLapRegFDValidation('uniform graph', 0)
rparam.dualLaplacianStencil.useUniformGraphWeights = False
dualLapRegFDValidation('non-uniform graph', 4)

rparam.dualLaplacianStencil.type = rparam.dualLaplacianStencil.Type.DualMeshIDT
dualLapRegFDValidation('dual IDT', 8)
plt.tight_layout(pad=0)
plt.show()

Validate that switching to the scale invariant fitting energy doesn't break anything. Note that finite differences do not behave particularly well on the fitting term because it is non-smooth. However, the behavior is the same with and without the scale invariant setting.

In [ ]:
def fdValidationFull(name, offset):
    perturb = np.random.uniform(low=-1,high=1, size=rparam.numVars())
    plt.subplot(2, 2, offset + 1)
    fd_validation.gradConvergencePlot(rparam, perturb=perturb, energyType=rparam.EnergyType.Fitting)
    plt.title(f'{name}, fitting gradient')
    plt.subplot(2, 2, offset + 2)
    fd_validation.hessConvergencePlot(rparam, perturb=perturb, energyType=rparam.EnergyType.Fitting)
    plt.title(f'{name}, fitting Hessian')
    
fig = plt.figure(figsize=(8, 6))
fdValidationFull('default', 0)
rparam.scaleInvariantFittingEnergy = True
fdValidationFull('scale invariant', 2)
plt.tight_layout(pad=0)
plt.show()

Check the scaling of the regularization terms under scaling of the input/UV and refinement.

We replace alpha with a fixed, smooth function to test the effect of refining the dual mesh on the measured energy. The DualMeshIDT is invariant to scaling and nearly invariant to subdivision. Surprisingly, the uniform graph Laplacian is also invariant to scaling and subdivison. However, the presumably better-behaved inverse length weight laplacian scales proportionally to 1 / averageEdgeLength. To achieve scale invariance, our implementation thus scales all weights by the average edge length.

### On the scaling of the uniform Laplacian
Assuming subdivision roughly preserves the proportion of boundary and interior edges, we can assume the number of edges scales proportionally to the number of faces:
6F ~= 2E ==> E ~= 3F.
Therefore, the number of edges scales like 1 / face_area, or 1 / edge_len^2.
Assuming a fine enough subdivision, the contribution of each edge to the uniform Laplacian Dirichlet energy scales like edge_len^2 (since the difference of values sampled at the edge endpoints should be proprotional to edge_len). Hence, the total Dirichlet energy scales like 1 / edge_len^2 * edge_len^2 = 1.

In [ ]:
def smoothTestFunction(inputMesh):
    V, F = inputMesh.vertices(), inputMesh.triangles()
    barycenters = V[F].mean(axis=1)
    diagLen = np.linalg.norm(barycenters.max(axis=0) -  barycenters.min(axis=0))
    barycenters *= 2.0 / diagLen
    return np.sin(2 * np.pi * barycenters[:, 0]) * np.sin(2 * np.pi * barycenters[:, 1]) * np.sin(2 * np.pi * barycenters[:, 2])

In [ ]:
def checkScaling(scale, nsubdiv, scaleInvariantFittingEnergy, useUniformGraphWeights, dualLaplacianType):
    upsampleMesh, upsampleUV = rparam.upsampledUV(nsubdiv)
    upsampleMesh.setVertices(upsampleMesh.vertices() * scale)
    upsampleUV *= scale
    rparam_upsample = parametrization.RegularizedParametrizerSVD(upsampleMesh, upsampleUV)
    rparam_upsample.alphaMin = alphaMin
    rparam_upsample.alphaMax = alphaMax
    rparam_upsample.bendRegW = 1.0
    
    rparam_upsample.dualLaplacianStencil.useUniformGraphWeights = rparam.dualLaplacianStencil.useUniformGraphWeights = useUniformGraphWeights
    rparam_upsample.dualLaplacianStencil.type = rparam.dualLaplacianStencil.type = dualLaplacianType
    rparam_upsample.scaleInvariantFittingEnergy = rparam.scaleInvariantFittingEnergy = scaleInvariantFittingEnergy
    
    rparam.setAlphas(smoothTestFunction(rparam.mesh()))
    rparam_upsample.setAlphas(smoothTestFunction(rparam_upsample.mesh()))
    
    return {name: rparam_upsample.energy(et) / rparam.energy(et) for name, et in rparam.EnergyType.__members__.items()}

In [ ]:
checkScaling(1.0, 2, scaleInvariantFittingEnergy=True, useUniformGraphWeights=True, dualLaplacianType=rparam.dualLaplacianStencil.Type.DualGraph)

In [ ]:
checkScaling(2.0, 0, scaleInvariantFittingEnergy=True, useUniformGraphWeights=False, dualLaplacianType=rparam.dualLaplacianStencil.Type.DualMeshIDT)

In [ ]:
checkScaling(2.0, 2, scaleInvariantFittingEnergy=True, useUniformGraphWeights=False, dualLaplacianType=rparam.dualLaplacianStencil.Type.DualMeshIDT)

In [ ]:
checkScaling(2.0, 2, scaleInvariantFittingEnergy=True, useUniformGraphWeights=False, dualLaplacianType=rparam.dualLaplacianStencil.Type.DualGraph)